In [ ]:
# FAST BERT QA (3–4 min)

!pip install -q transformers datasets torch

import torch
from datasets import load_dataset
from transformers import BertTokenizerFast, BertForQuestionAnswering, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 🔥 Only 800 samples
dataset = load_dataset("squad")
train_dataset = dataset["train"].select(range(800))

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def preprocess(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )
    inputs["start_positions"] = [0]*len(examples["question"])
    inputs["end_positions"] = [1]*len(examples["question"])
    return inputs

train_dataset = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
train_dataset.set_format("torch")

model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model.to(device)

training_args = TrainingArguments(
    output_dir="./fast_qa",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)

trainer.train()

# Test
question = "Who developed BERT?"
context = "BERT was developed by researchers at Google in 2018."

inputs = tokenizer(question, context, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

start = torch.argmax(outputs.start_logits)
end = torch.argmax(outputs.end_logits) + 1

answer = tokenizer.decode(inputs["input_ids"][0][start:end])
print("Answer:", answer)

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,0.705995


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: [CLS] who
